# 🧰 **Utils & Imports**

In [1]:
!pip install -q -U transformers accelerate peft
!pip install datasets==2.21.0
!pip install -q --no-build-isolation seqeval
!pip install evaluate==0.4.3 seqeval==1.2.2

In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer)
from peft import LoraConfig,TaskType,get_peft_model
import evaluate
from peft import PeftModel

# 📘 **Dataset**

In [3]:
dataset = load_dataset("HaniehPoostchi/persian_ner", trust_remote_code=True)
dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 5121
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2560
    })
})

In [4]:
sample = dataset["train"][331]
print(sample)

{'tokens': ['آقای', 'خاتمی', 'تأکید', 'کرد', ':', 'هماهنگ', 'کردن', 'خواستها', 'و', 'نیازهای', 'ملت', 'با', 'توانائیها', 'و', 'امکانات', 'مسئولان', 'کار', 'بسیار', 'سختی', 'است', 'كه', 'رسانه\u200cها', 'بر', 'عهده', 'دارند', '.'], 'ner_tags': [0, 11, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [5]:
labels = dataset["train"].features["ner_tags"].feature.names
labels

['O',
 'I-event',
 'I-fac',
 'I-loc',
 'I-org',
 'I-pers',
 'I-pro',
 'B-event',
 'B-fac',
 'B-loc',
 'B-org',
 'B-pers',
 'B-pro']

# 🔐 **Tokenizer**

In [6]:
model_checkpoint = "HooshvareLab/bert-base-parsbert-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [7]:
tokens = dataset["train"][0]["tokens"]
encoded = tokenizer(tokens,is_split_into_words=True)        # already words splited
print(encoded.tokens())

['[CLS]', 'به', 'عنوان', 'مثال', 'وقتی', 'نشریات', 'مدافع', 'اصول', 'و', 'ارزشها', 'و', 'منادی', 'انقلاب', 'و', 'اسلام', 'در', 'بالاترین', 'درجه', '،', 'اولین', 'و', 'درشتترین', 'تیتر', 'نشریه', 'خود', 'را', 'در', 'صدر', 'صفحه', 'نخستین', '،', 'به', 'تکذیب', 'اظهارات', 'و', 'نظریات', 'مشاور', 'ريیسجمهور', 'با', 'همین', 'ترکیب', 'عبارتی', 'و', 'البته', 'از', 'قول', 'دیگران', 'اختصاص', 'میدهند', '،', 'ایا', 'در', 'موارد', 'مشابه', 'نیز', 'هر', 'گاه', 'خبر', 'تکذیب', 'متوجه', 'و', 'معطوف', 'به', 'شخصی', 'باشد', '[UNK]', 'در', 'زمره', 'مشاوران', '[UNK]', 'مقام', 'بلندمرتبه', '##ی', 'دیگر', 'است', '،', 'خبر', 'را', 'عینا', 'به', 'همین', 'درشتی', 'و', 'با', 'همین', 'ترکیب', 'عبارتی', 'در', 'صدر', 'صفحه', 'نخست', 'به', 'چاپ', 'میرساند', 'و', 'در', 'ان', 'مورد', 'هم', 'به', 'جای', 'ذکر', 'نام', 'ي', '##ا', 'عضویت', 'ان', 'شخص', 'در', 'گروه', 'و', 'کمیتهی', 'خاص', 'صرفا', 'بر', 'روی', 'عنوان', 'مشاور', 'فلان', 'مسيول', 'بلندمرتبه', 'تاکید', 'میکنند', '؟', 'اگر', 'نشریه', 'دیگری', 'چنین', 'کند',

In [8]:
label_names = dataset["train"].features["ner_tags"].feature.names     # Mapping label
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}
print(id2label)

{0: 'O', 1: 'I-event', 2: 'I-fac', 3: 'I-loc', 4: 'I-org', 5: 'I-pers', 6: 'I-pro', 7: 'B-event', 8: 'B-fac', 9: 'B-loc', 10: 'B-org', 11: 'B-pers', 12: 'B-pro'}


In [9]:
def tokenize_and_align_labels(example):
    'Heart of NER'
    tokenized_inputs = tokenizer(example["tokens"],truncation=True,is_split_into_words=True)
    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None
    for word_idx in word_ids:

        if word_idx is None:
            labels.append(-100)

        elif word_idx != previous_word_idx:
            labels.append(example["ner_tags"][word_idx])

        else:
            labels.append(example["ner_tags"][word_idx])
        previous_word_idx = word_idx
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [10]:
tokenized_dataset = dataset.map(tokenize_and_align_labels)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5121
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2560
    })
})

In [11]:
data_collator=DataCollatorForTokenClassification(tokenizer)
seqeval=evaluate.load("seqeval")

In [12]:
def compute_metrics(p):
    predictions,labels=p
    predictions=np.argmax(predictions,axis=2)
    true_predictions=[[label_names[p] for p,l in zip(prediction,label) if l!=-100] for prediction,label in zip(predictions,labels)]
    true_labels=[[label_names[l] for p,l in zip(prediction,label) if l!=-100] for prediction,label in zip(predictions,labels)]
    results=seqeval.compute(predictions=true_predictions,references=true_labels)
    return {
        "precision":results["overall_precision"],
        "recall":results["overall_recall"],
        "f1":results["overall_f1"],
        "accuracy":results["overall_accuracy"]}

# 🧠 **Model**

In [ ]:
model=AutoModelForTokenClassification.from_pretrained(model_checkpoint ,num_labels=len(label_names) ,id2label=id2label ,label2id=label2id)
model

# 🔋 **Full Fine-tune**

In [14]:
args=TrainingArguments(
    output_dir="./Ner-full",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1)

In [15]:
trainer=Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.031297,0.105451,0.766388,0.819611,0.792106,0.973533
2,0.017244,0.109859,0.783548,0.823761,0.803151,0.975357
3,0.008881,0.117507,0.787367,0.824853,0.805674,0.975727


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=963, training_loss=0.020633064194941198, metrics={'train_runtime': 165.7998, 'train_samples_per_second': 92.66, 'train_steps_per_second': 5.808, 'total_flos': 634348708604592.0, 'train_loss': 0.020633064194941198, 'epoch': 3.0})

In [18]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.008881,0.117507,3,0.787367,0.824853,0.805674,0.975727


{'eval_loss': 0.11750689893960953,
 'eval_precision': 0.7873671044402751,
 'eval_recall': 0.8248525879012885,
 'eval_f1': 0.8056740614334471,
 'eval_accuracy': 0.9757269399969003}

In [19]:
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
total=sum(p.numel() for p in model.parameters())
print(trainable)
print(total)
print(f"{100*trainable/total:.2f}%")

162260749
162260749
100.00%


In [20]:
model_full=AutoModelForTokenClassification.from_pretrained("./Ner-full/checkpoint-963")
model_full.save_pretrained("/content/gdrive/MyDrive/NLP_Models/Ner-full-fine-tune")
tokenizer.save_pretrained("/content/gdrive/MyDrive/NLP_Models/Ner-full-fine-tune")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/gdrive/MyDrive/NLP_Models/Ner-full-fine-tune/tokenizer_config.json',
 '/content/gdrive/MyDrive/NLP_Models/Ner-full-fine-tune/tokenizer.json')

# 📊 **LoRA**

In [ ]:
!pip install -U torchao

In [ ]:
model_lora=AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id)

In [23]:
lora_config=LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=32,             # more parameter ,power ,memory
    lora_alpha=64,
    target_modules=["query","key","value"],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["classifier"])

In [26]:
model_lora=get_peft_model(
    model_lora,
    lora_config)                # Change to peft

In [27]:
model_lora.print_trainable_parameters()          # compare with ,after full Fine-tune

trainable params: 1,779,469 || all params: 164,040,218 || trainable%: 1.0848


In [28]:
args_lora=TrainingArguments(
    output_dir="./Ner-Lora-v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1)

In [29]:
trainer_lora=Trainer(
    model=model_lora,
    args=args_lora,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics)

In [30]:
trainer_lora.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.169040,0.153659,0.562669,0.591177,0.576571,0.956068
2,0.124594,0.119565,0.662816,0.676567,0.669621,0.965391
3,0.101941,0.113428,0.670680,0.693383,0.681843,0.966702


TrainOutput(global_step=963, training_loss=0.21778991413017423, metrics={'train_runtime': 98.1565, 'train_samples_per_second': 156.515, 'train_steps_per_second': 9.811, 'total_flos': 647618450945376.0, 'train_loss': 0.21778991413017423, 'epoch': 3.0})

In [31]:
trainer_lora.evaluate()

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.101941,0.113428,3,0.670680,0.693383,0.681843,0.966702


{'eval_loss': 0.1134282574057579,
 'eval_precision': 0.6706801858893113,
 'eval_recall': 0.6933828346800611,
 'eval_f1': 0.6818425856329862,
 'eval_accuracy': 0.9667020350743333}

In [32]:
trainer_lora.model.save_pretrained("/content/gdrive/MyDrive/NLP_Models/Ner-Lora-v2")
tokenizer.save_pretrained("/content/gdrive/MyDrive/NLP_Models/Ner-Lora-v2")

('/content/gdrive/MyDrive/NLP_Models/Ner-Lora-v2/tokenizer_config.json',
 '/content/gdrive/MyDrive/NLP_Models/Ner-Lora-v2/tokenizer.json')

# ✅ **BestLoRA**

In [ ]:
model_lora_v3=AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

In [34]:
lora_config_v3=LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=16,
    lora_alpha=32,
    target_modules=[
        "query",
        "key",
        "value",
        "intermediate.dense",
        "output.dense"
    ],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["classifier"]
)

In [35]:
model_lora_v3=get_peft_model(
    model_lora_v3,
    lora_config_v3
)

model_lora_v3.print_trainable_parameters()

trainable params: 2,664,205 || all params: 164,924,954 || trainable%: 1.6154


In [36]:
args_lora_v3=TrainingArguments(
    output_dir="./Ner-Lora-v3",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1
)

In [37]:
trainer_lora_v3=Trainer(
    model=model_lora_v3,
    args=args_lora_v3,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [38]:
trainer_lora_v3.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.158811,0.143193,0.584080,0.612142,0.597782,0.958750
2,0.103910,0.106758,0.706681,0.716095,0.711357,0.968502
3,0.082646,0.098472,0.731142,0.757807,0.744236,0.970589
4,0.076986,0.095362,0.735993,0.777462,0.756160,0.971638
5,0.065562,0.094167,0.746423,0.774623,0.760261,0.972019


TrainOutput(global_step=1605, training_loss=0.16092898680413623, metrics={'train_runtime': 237.1059, 'train_samples_per_second': 107.99, 'train_steps_per_second': 6.769, 'total_flos': 1091403016010496.0, 'train_loss': 0.16092898680413623, 'epoch': 5.0})

In [39]:
model_lora_v3.print_trainable_parameters()              # only on 2.5 milion parameter (1.6%)

trainable params: 2,664,205 || all params: 164,924,954 || trainable%: 1.6154


In [40]:
trainer_lora_v3.evaluate()

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.065562,0.094167,5,0.746423,0.774623,0.760261,0.972019


{'eval_loss': 0.09416709095239639,
 'eval_precision': 0.7464225589225589,
 'eval_recall': 0.7746232801921817,
 'eval_f1': 0.7602614939449147,
 'eval_accuracy': 0.9720192181594917}

In [41]:
trainer_lora_v3.model.save_pretrained("/content/gdrive/MyDrive/NLP_Models/Ner-Lora-best")
tokenizer.save_pretrained("/content/gdrive/MyDrive/NLP_Models/Ner-Lora-best")

('/content/gdrive/MyDrive/NLP_Models/Ner-Lora-best/tokenizer_config.json',
 '/content/gdrive/MyDrive/NLP_Models/Ner-Lora-best/tokenizer.json')

# 🆎 **Prediction & Compare Full 🆚 Lora**

In [42]:
def predict_ner(model,text):
    model.eval()
    inputs=tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs=model(**inputs)

    predictions=torch.argmax(outputs.logits,dim=2)[0].cpu().numpy()
    tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    results=[]
    for token,pred in zip(tokens,predictions):
        if token in ["[CLS]","[SEP]","[PAD]"]:
            continue
        results.append((token,id2label[pred]))

    return results

In [ ]:
text="محمد خاتمی در تهران با مسئولان سازمان ملل دیدار کرد"

model_full = AutoModelForTokenClassification.from_pretrained("/content/gdrive/MyDrive/NLP_Models/Ner-full-fine-tune")
model_full.to("cuda")

In [50]:
predict_ner(model,text)

[('محمد', 'B-pers'),
 ('خاتمی', 'I-pers'),
 ('در', 'O'),
 ('تهران', 'B-loc'),
 ('با', 'O'),
 ('مسيولان', 'O'),
 ('سازمان', 'B-org'),
 ('ملل', 'I-org'),
 ('دیدار', 'O'),
 ('کرد', 'O')]

In [ ]:
base_model=AutoModelForTokenClassification.from_pretrained(
model_checkpoint,
num_labels=len(label_names),
id2label=id2label,
label2id=label2id
)

model_lora_best_loaded=PeftModel.from_pretrained(base_model,"/content/gdrive/MyDrive/NLP_Models/Ner-Lora-best")
model_lora_best_loaded.to("cuda")

In [52]:
predict_ner(model_lora_best_loaded.model,text)

[('محمد', 'B-pers'),
 ('خاتمی', 'I-pers'),
 ('در', 'O'),
 ('تهران', 'B-loc'),
 ('با', 'O'),
 ('مسيولان', 'O'),
 ('سازمان', 'B-org'),
 ('ملل', 'I-org'),
 ('دیدار', 'O'),
 ('کرد', 'O')]

In [54]:
label_name={
"pers":"شخص",
"loc":"مکان",
"org":"سازمان",
"event":"رویداد",
"fac":"ساختمان/تأسیسات",
"pro":"محصول"}

def merge_entities(result):
    entities=[]
    current_word=[]
    current_label=None

    for word,label in result:
        if label=="O":
            if current_word:
                entities.append((" ".join(current_word),current_label))
                current_word=[]
                current_label=None
            continue

        prefix,entity=label.split("-")

        if prefix=="B":
            if current_word:
                entities.append((" ".join(current_word),current_label))

            current_word=[word]
            current_label=label_name[entity]

        elif prefix=="I":
            if current_label==label_name[entity]:
                current_word.append(word)
            else:
                if current_word:
                    entities.append(
                        (" ".join(current_word),current_label))
                current_word=[word]
                current_label=label_name[entity]

    if current_word:
        entities.append((" ".join(current_word),current_label))

    return entities

In [55]:
texts=[
"علی رضایی مدیر شرکت مایکروسافت در تهران با مسئولان وزارت ارتباطات ایران دیدار کرد.",
"دکتر مریم احمدی در دانشگاه تهران درباره هوش مصنوعی سخنرانی کرد.",
"شرکت اپل در سال 2025 گوشی آیفون 17 را در آمریکا معرفی کرد.",
"تیم ملی فوتبال ایران در جام جهانی 2022 مقابل تیم ملی انگلیس بازی کرد.",
"سازمان ملل متحد در نیویورک جلسه‌ای درباره تغییرات اقلیمی برگزار کرد.",
"شرکت سامسونگ کارخانه جدید خود را در کره جنوبی افتتاح کرد.",
"برج میلاد تهران یکی از معروف‌ترین مراکز گردشگری ایران است.",
"ایلان ماسک مدیر شرکت تسلا اعلام کرد که خودروی جدید این شرکت به زودی عرضه می‌شود.",
"دانشگاه صنعتی شریف و دانشگاه تهران از برترین مراکز علمی ایران هستند.",
"مراسم المپیک 2024 پاریس با حضور ورزشکاران کشورهای مختلف برگزار شد."
]

In [56]:
for text in texts:
    print("\nTEXT:",text)
    print("FULL:",merge_entities(predict_ner(model_full,text)))
    print("LORA:",merge_entities(predict_ner(model_lora_best_loaded,text)))


TEXT: علی رضایی مدیر شرکت مایکروسافت در تهران با مسئولان وزارت ارتباطات ایران دیدار کرد.
FULL: [('علی رضایی', 'شخص'), ('شرکت مایکروسافت', 'سازمان'), ('تهران', 'مکان'), ('وزارت ارتباطات ایران', 'سازمان')]
LORA: [('علی رضایی', 'شخص'), ('شرکت مایکروسافت', 'سازمان'), ('تهران', 'مکان'), ('وزارت ارتباطات ایران', 'سازمان')]

TEXT: دکتر مریم احمدی در دانشگاه تهران درباره هوش مصنوعی سخنرانی کرد.
FULL: [('مریم احمدی', 'شخص'), ('دانشگاه تهران', 'ساختمان/تأسیسات')]
LORA: [('مریم احمدی', 'شخص'), ('دانشگاه تهران', 'ساختمان/تأسیسات')]

TEXT: شرکت اپل در سال 2025 گوشی آیفون 17 را در آمریکا معرفی کرد.
FULL: [('شرکت اپل', 'سازمان'), ('امریکا', 'مکان')]
LORA: [('شرکت اپل', 'سازمان'), ('ایفون', 'محصول'), ('امریکا', 'مکان')]

TEXT: تیم ملی فوتبال ایران در جام جهانی 2022 مقابل تیم ملی انگلیس بازی کرد.
FULL: [('تیم ملی فوتبال ایران', 'سازمان'), ('جام جهانی [UNK]', 'رویداد'), ('تیم ملی انگلیس', 'سازمان')]
LORA: [('تیم ملی فوتبال ایران', 'سازمان'), ('جام جهانی [UNK]', 'رویداد'), ('تیم ملی انگلیس', 'سازمان')]


In [58]:
def compare_entity_agreement(model1,model2,dataset):          # Compare Without O lables
    model1.eval()
    model2.eval()
    same=0
    total=0

    for item in dataset:
        inputs={
            "input_ids":torch.tensor([item["input_ids"]]).to("cuda"),
            "attention_mask":torch.tensor([item["attention_mask"]]).to("cuda")}

        with torch.no_grad():
            pred1=model1(**inputs).logits.argmax(-1).cpu().numpy()[0]
            pred2=model2(**inputs).logits.argmax(-1).cpu().numpy()[0]
        labels=np.array(item["labels"])
        mask=(labels!=-100)&(labels!=0)

        same+=(pred1[mask]==pred2[mask]).sum()
        total+=mask.sum()
    return same/total

entity_agreement=compare_entity_agreement(model_full, model_lora_best_loaded, tokenized_dataset["test"])
print(f"Entity Agreement:{entity_agreement*100:.2f}%")

'91.15% similiar full Fine-tune (without O lables ,only on entities for more Efficiency)'

Entity Agreement:91.15%


'91.15% similiar full Fine-tune (without O lables ,only on entities for more Efficiency)'

In [59]:
!du -sh /content/gdrive/MyDrive/NLP_Models/Ner-full-fine-tune
!du -sh /content/gdrive/MyDrive/NLP_Models/Ner-Lora-best

622M	/content/gdrive/MyDrive/NLP_Models/Ner-full-fine-tune
13M	/content/gdrive/MyDrive/NLP_Models/Ner-Lora-best
